# `03-geocoding-umpd-incidents.ipynb` : Using Geocodio.io to geocode as many locations we can without having to rely on Google

## Setup

In [1]:
import pandas as pd 
import requests
import os
import json
import time
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"
CURRENT_YEAR = time.localtime().tm_year

GEOCODIO_BASE_URL = "https://api.geocod.io/v2.1/geocode/"

HEADERS = {
  "User-Agent": "Jour720/1.0 (diegojto@umd.edu)",
  "Accept-Language": "en-US,en;q=0.9",
}

geocodio_params = {
  "q": None,
  "api_key" : os.getenv("UMPDCRIMEGEOCODIOAPIKEY")
}

df = pd.read_csv(DATA_DIR / "processed" / f"merged_arrest_ci_data_2018-{CURRENT_YEAR}.csv")

In [2]:
if not Path(DATA_DIR / "processed" / "geocoding").is_dir():
  os.mkdir(Path(DATA_DIR / "processed" / "geocoding"))
  geocoding_dir = Path(DATA_DIR / "processed" / "geocoding")
else:
  i = 1
  while Path(DATA_DIR / "processed" / f"geocoding_{i}").is_dir():
    i += 1
  os.mkdir(Path(DATA_DIR / "processed" / f"geocoding_{i}"))
  geocoding_dir = Path(DATA_DIR / "processed" / f"geocoding_{i}" )

In [3]:
cases_and_locs_df = df[['umpd_case_number', 'location']].drop_duplicates()
cases_and_locs_df.to_csv(geocoding_dir / "ci_locs.csv")

In [8]:
cases_and_locs_df

,umpd_case_number,location
0,2019-00000001,4300 block of Knox Rd
1,2019-00000009,8300 block of Baltimore Ave
2,2019-00000011,7400 block of Baltimore Ave
3,2019-00000203,7500 block of Calvert Service Ln
4,2019-00000312,Metzerott Rd
...,...,...
13260,2026-00289088,"Frat 2, Sigma Alpha Mu, Sigma Alpha Mu Fratern..."
13261,2026-00289157,Baltimore Ave
13262,2026-00289166,8600 block of Boteler Ln
13263,2026-00289167,3900 block of Campus Dr


In [ ]:
response = requests.get(GEOCODIO_BASE_URL, params={
  "q": "4300 block of Knox Rd, College Park, MD",
  "api_key": os.getenv("UMPDCRIMEGEOCODIOAPIKEY")
}, headers = HEADERS)

response.json()

In [21]:
df.head()

,Unnamed: 0,umpd_case_number,date_occurred,date_occurred_tod,date_occurred_sem,date_occurred_sem_season,report_date,report_date_tod,report_date_sem,report_date_sem_season,...,arrest_number,arrest_date,arrest_date_semester,arrest_date_semester_season,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges
0,0,2019-00000001,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,...,21888.0,2019-01-01 00:01:00,Not during a semester,Not during a semester,NaN,Asian/Pacific Islander,Male,Fireworks Discharge without the required permit,1.0,['Fireworks Discharge without the required per...
1,1,2019-00000009,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,...,21895.0,2019-01-01 01:37:00,Not during a semester,Not during a semester,NaN,Black,Female,"(Driving, Attempting to drive) veh. while impa...",6.0,"['(Driving, Attempting to drive) veh. while im..."
2,2,2019-00000011,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,...,21896.0,2019-01-01 01:41:00,Not during a semester,Not during a semester,NaN,Asian/Pacific Islander,Male,"(Driving, Attempting to drive) veh. while impa...",2.0,"['(Driving, Attempting to drive) veh. while im..."
3,3,2019-00000203,2018-12-28 17:00:00,afternoon,Not during a semester,Not during a semester,2019-01-02 11:34:00,morning,Winter 2019,Not during a semester,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2019-00000312,2019-01-02 23:04:00,evening,Winter 2019,Winter,2019-01-02 23:04:00,evening,Winter 2019,Winter,...,21889.0,2019-01-02 23:30:00,Winter 2019,Winter,NaN,Black,Male,CDS:Possession Marijuana L/T 10 grams,1.0,['CDS:Possession Marijuana L/T 10 grams']


In [ ]:
df['arrest_date'] = pd.to_datetime(df['arrest_date'])
df_2025 = df[df['arrest_date'].dt.year == 2025]
cases_and_locs_2025 = df_2025[['umpd_case_number', 'location']]

,umpd_case_number,location
3976,2021-00081808,"64, Dorchester, Dorchester Hall at 3947 Campus Dr"
6384,2023-00038780,Baltimore Ave
6624,2023-00055167,"Laboratory for Physical Sciences, LPS at 8050 ..."
6893,2023-00079735,"A James Clark Hall, A. James Clark Hall, A.J. ..."
7058,2023-00086320,"Commons 4, SCC 4, South Campus Commons 4 at 43..."
...,...,...
11550,2025-00283429,University Blvd
11551,2025-00283504,Baltimore Ave
11561,2025-00286752,Baltimore Ave
11576,2025-00290080,University Blvd


In [ ]:
results = []
start_time = time.time()
for i, case_num, location in tuple(cases_and_locs_2025.itertuples()):
  
  result_dict = {"umpd_case_number": case_num, "source_location": location + " College Park, MD"}
  
  if i == 0: 
    print("Beginning geocoding")
  elif i == len(cases_and_locs_df) // 2:
    print("Halfway done")
  geocodio_params['q'] = location + " College Park, MD"
  response = requests.get(GEOCODIO_BASE_URL, params=geocodio_params, headers=HEADERS)
  if response.status_code != 200: 
    print(f"Error geocoding {location}, setting to None")
    print(f"Received {response.status_code}, saying {response.reason}")
    result_dict['geocodio_formatted_address'] = None
    result_dict['geocodio_lat'] = None
    result_dict['geocodio_long'] = None
    result_dict['geocodio_accuracy'] = None
    result_dict['geocodio_accuracy_type'] = None
    result_dict['geocodio_match_type'] = None
  else: 
    data = response.json()
    try:
      data = data['results'][0]
    except IndexError as e:
      print("Didn't return expected structure:")
      print(data['results'])
      print("Setting options to None")
      result_dict['geocodio_formatted_address'] = None
      result_dict['geocodio_lat'] = None
      result_dict['geocodio_long'] = None
      result_dict['geocodio_accuracy'] = None
      result_dict['geocodio_accuracy_type'] = None
      result_dict['geocodio_match_type'] = None
    else: 
      result_dict['geocodio_formatted_address'] = data['formatted_address'] if data['formatted_address'] else "",
      result_dict["geocodio_lat"] = data['location']['lat'] if data['location']['lat'] else 0.0,
      result_dict["geocodio_long"] = data['location']['lng'] if data['location']['lng'] else 0.0,
      result_dict['geocodio_accuracy'] = data['accuracy'] if data['accuracy'] else 0,
      result_dict['geocodio_accuracy_type'] = data['accuracy_type'] if data['accuracy_type'] else "",
      result_dict['geocodio_match_type'] = data['match_type'] if data['match_type'] else ""
    
  results.append(result_dict)
print("Done")
end_time = time.time()
execution_time = end_time - start_time
print(f"Geocoding {len(cases_and_locs_2025)} addresses took {execution_time} seconds")

Done
Geocoding 12912 addresses took 92.83824825286865 seconds


In [31]:
geocodio_2025_results_df = pd.DataFrame(results)
geocodio_2025_results_df

,umpd_case_number,source_location,geocodio_formatted_address,geocodio_lat,geocodio_long,geocodio_accuracy,geocodio_accuracy_type,geocodio_match_type
0,2021-00081808,"64, Dorchester, Dorchester Hall at 3947 Campus...","(64 West Dr, College Park, MD 20740,)","(39.029713,)","(-76.941253,)","(0.87,)","(range_interpolation,)",
1,2023-00038780,"Baltimore Ave College Park, MD","(Baltimore Ave, College Park, MD 20740,)","(39.00021,)","(-76.93121,)","(1,)","(street_center,)",
2,2023-00055167,"Laboratory for Physical Sciences, LPS at 8050 ...","(8050 Greenmead Dr, College Park, MD 20740,)","(39.00349,)","(-76.943676,)","(1,)","(rooftop,)",building_centroid
3,2023-00079735,"A James Clark Hall, A. James Clark Hall, A.J. ...","(Paint Branch Dr, College Park, MD 20740,)","(38.998712,)","(-76.941425,)","(1,)","(street_center,)",
4,2023-00086320,"Commons 4, SCC 4, South Campus Commons 4 at 43...","(4 4th St, Laurel, MD 20707,)","(39.107653,)","(-76.846698,)","(0.7,)","(range_interpolation,)",
...,...,...,...,...,...,...,...,...
422,2025-00283429,"University Blvd College Park, MD","(University Blvd, College Park, MD 20740,)","(39.00009,)","(-76.943231,)","(1,)","(street_center,)",
423,2025-00283504,"Baltimore Ave College Park, MD","(Baltimore Ave, College Park, MD 20740,)","(39.00021,)","(-76.93121,)","(1,)","(street_center,)",
424,2025-00286752,"Baltimore Ave College Park, MD","(Baltimore Ave, College Park, MD 20740,)","(39.00021,)","(-76.93121,)","(1,)","(street_center,)",
425,2025-00290080,"University Blvd College Park, MD","(University Blvd, College Park, MD 20740,)","(39.00009,)","(-76.943231,)","(1,)","(street_center,)",


In [32]:
df.head()

,Unnamed: 0,umpd_case_number,date_occurred,date_occurred_tod,date_occurred_sem,date_occurred_sem_season,report_date,report_date_tod,report_date_sem,report_date_sem_season,...,arrest_number,arrest_date,arrest_date_semester,arrest_date_semester_season,age,race,sex,arrested_charge,arrested_charges_num,arrested_charges
0,0,2019-00000001,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,2019-01-01 00:01:00,night,Not during a semester,Not during a semester,...,21888.0,2019-01-01 00:01:00,Not during a semester,Not during a semester,NaN,Asian/Pacific Islander,Male,Fireworks Discharge without the required permit,1.0,['Fireworks Discharge without the required per...
1,1,2019-00000009,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,2019-01-01 01:20:00,night,Not during a semester,Not during a semester,...,21895.0,2019-01-01 01:37:00,Not during a semester,Not during a semester,NaN,Black,Female,"(Driving, Attempting to drive) veh. while impa...",6.0,"['(Driving, Attempting to drive) veh. while im..."
2,2,2019-00000011,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,2019-01-01 01:28:00,night,Not during a semester,Not during a semester,...,21896.0,2019-01-01 01:41:00,Not during a semester,Not during a semester,NaN,Asian/Pacific Islander,Male,"(Driving, Attempting to drive) veh. while impa...",2.0,"['(Driving, Attempting to drive) veh. while im..."
3,3,2019-00000203,2018-12-28 17:00:00,afternoon,Not during a semester,Not during a semester,2019-01-02 11:34:00,morning,Winter 2019,Not during a semester,...,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,2019-00000312,2019-01-02 23:04:00,evening,Winter 2019,Winter,2019-01-02 23:04:00,evening,Winter 2019,Winter,...,21889.0,2019-01-02 23:30:00,Winter 2019,Winter,NaN,Black,Male,CDS:Possession Marijuana L/T 10 grams,1.0,['CDS:Possession Marijuana L/T 10 grams']
